<img src="https://hilpisch.com/tce_logo.png" alt="The Crypto Engineer" width="25%" align="right" border="0"><br>


# Bitcoin Foundations
## Chapter 2 – Hashes, Addresses, and Identities

&copy; Dr. Yves J. Hilpisch | The Python Quants GmbH<br>
The Crypto Engineer | AI-Powered by GPT 5.1.


You begin by turning a toy compressed public key into a shorter fingerprint using SHA-256 followed by RIPEMD-160. This mirrors the real "hash160" pipeline used in classic Bitcoin addresses and gives you a concrete feel for how long public keys become fixed-size identifiers.

In [1]:
import hashlib  # hashing primitives from the standard library

pubkey_hex = "02" + "11" * 32  # toy compressed public key (not real)
pubkey_bytes = bytes.fromhex(pubkey_hex)  # parse hex into bytes

sha256_digest = hashlib.sha256(pubkey_bytes).digest()  # SHA-256 of public key
ripemd160 = hashlib.new("ripemd160")  # construct RIPEMD-160 hash object
ripemd160.update(sha256_digest)  # feed SHA-256 output into RIPEMD-160
hash160_bytes = ripemd160.digest()  # 20-byte fingerprint

print("pubkey:", pubkey_hex)  # show toy public key
print("hash160:", hash160_bytes.hex())  # show resulting fingerprint


pubkey: 021111111111111111111111111111111111111111111111111111111111111111
hash160: adfce54f529b2154e3c361bbe3f7d41db0635717


Next you wrap this fingerprint in a Version+Payload+Checksum structure and encode it with a Base58-like alphabet. This toy Base58Check sketch leaves out edge cases, but it makes visible how version bytes, payloads, and checksums combine into the address strings you see in wallets and explorers.

In [2]:
import hashlib  # reused for checksum

alphabet = "123456789ABCDEFGHJKLMNPQRSTUVWXYZabcdefghijkmnopqrstuvwxyz"  # Base58 set

version_byte = b"\x00"  # pretend mainnet P2PKH
payload = hash160_bytes  # use fingerprint from previous example
data = version_byte + payload  # version + payload

checksum_full = hashlib.sha256(hashlib.sha256(data).digest()).digest()  # double SHA-256
checksum = checksum_full[:4]  # first four bytes as checksum
full = data + checksum  # bytes to encode

num = int.from_bytes(full, "big")  # interpret as big-endian integer
encoded = ""  # result string
while num > 0:  # repeated division by 58
    num, rem = divmod(num, 58)  # quotient and remainder
    encoded = alphabet[rem] + encoded  # prepend matching character

print("Toy address:", encoded)  # human-readable representation


Toy address: Grxrh4z458DEdd3VVS6D5ASSyt8ydPaLe


<img src="https://hilpisch.com/tce_logo.png" alt="The Crypto Engineer" width="25%" align="right" border="0"><br>
